# Aprendizado de Máquina — Lista prática 02

## Regressão Linear e Regularização

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

O `superconductivity.csv` tem 21 263 supercondutores e 81 covariáveis. Com todas
as observações, $n$ é 260 vezes maior que $d$ e o mínimos quadrados vai muito bem
— não há o que regularizar. Esta lista faz o contrário: fica com **100
observações de treino**, para pôr você exatamente no regime em que a Aula 02
mora, com $n$ pouco maior que $d$.

> **quando $n$ e $d$ são comparáveis, o mínimos quadrados não fica só um pouco
> pior — ele quebra.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — os dados e a divisão

A célula de carga é a mesma da aula prática: ela não baixa nada da internet,
procura o `superconductivity.csv` na pasta deste notebook e, se você tiver o
repositório do curso, em `recursos/dados/`. Complete a separação da resposta e
a divisão treino/teste.

A resposta é `critical_temp`, a temperatura crítica em kelvin.

In [ ]:
_nome = "superconductivity.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../../recursos/dados/
_lugares = [_nome, os.path.join("..", "..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"nao encontrei '{_nome}'. Procurei nesta pasta e em "
        "../../recursos/dados/. Ponha o .csv ao lado deste notebook, "
        "ou mude o caminho se for necessário."
    )

df = pd.read_csv(_caminho)
X_todos = df.drop(columns="critical_temp")                    # (a)
y_todos = df["critical_temp"].values                          # (b)

print("dimensoes:", X_todos.shape)

Agora a divisão. Peça **100** observações de treino e 5 000 de teste — o resto do
banco fica de fora de propósito. A semente é 2026.

In [ ]:
X_tr, X_te, y_tr, y_te = skm.train_test_split(
    X_todos.values, y_todos,
    train_size=100,                                           # (a)
    test_size=5000,
    random_state=2026,                                        # (b)
)

print(f"treino: {X_tr.shape[0]} observacoes, {X_tr.shape[1]} covariaveis")
print(f"teste:  {X_te.shape[0]} observacoes")

Deve imprimir `dimensoes: (21263, 81)`, depois `treino: 100 observacoes, 81
covariaveis` e `teste: 5000 observacoes`.

Guarde a razão: **100 observações para estimar 82 parâmetros** (81 coeficientes
mais o intercepto). Sobram 18 graus de liberdade.

---
## Exercício 2 — o mínimos quadrados quebra

Ajuste o MQO e meça o $R^2$ nos **dois** conjuntos. O `Pipeline` padroniza antes
de ajustar; para o MQO isso não muda nada (Exercício 2(c) da lista teórica), mas
deixa o código pronto para o Ridge e o Lasso.

In [ ]:
def tubo(modelo):
    return Pipeline([("escala", StandardScaler()), ("mod", modelo)])


mqo = tubo(skl.LinearRegression()).fit(X_tr, y_tr)            # (a)

print(f"MQO   R2 treino {mqo.score(X_tr, y_tr):.4f}")
print(f"MQO   R2 teste  {mqo.score(X_te, y_te):.4f}")         # (b)

Deve imprimir `R2 treino 0.9582` e `R2 teste -19.0886`.

O modelo explica **95,8% da variação no treino** e tem $R^2$ **negativo** no
teste. Não é um detalhe: um $R^2$ de $-19$ significa que o erro quadrático do
modelo é 20 vezes o erro de quem chuta sempre a média. Em kelvin: o MQO erra em
média 153,8 K, e chutar a média do treino erraria 34,6 K.

**Responda** na célula abaixo, como comentário: o que significa um $R^2$
negativo, e por que ele apareceu aqui?

$R^2 = 1 - \mathrm{SQE}/\mathrm{SQT}$, onde SQT é a soma de quadrados em torno da
média. $R^2<0$ quer dizer $\mathrm{SQE}>\mathrm{SQT}$: **o modelo erra mais do que
o preditor constante**. Isso é impossível no conjunto de treino de uma regressão
com intercepto (o MQO poderia sempre zerar os outros coeficientes e recuperar a
média), mas é perfeitamente possível fora dele.

Apareceu aqui porque 82 parâmetros estimados com 100 observações é quase
interpolação: cada coeficiente é estimado com enorme incerteza, e as covariáveis
do banco são altamente correlacionadas entre si, o que deixa
$X^\top X$ mal-condicionada. Coeficientes gigantes e de sinais opostos
se cancelam nos pontos de treino e explodem em qualquer ponto novo.

---
## Exercício 3 — Ridge e Lasso no mesmo treino

Sem trocar uma única observação, ajuste os dois métodos penalizados. Use
$\alpha=10$ na Ridge e $\alpha=1$ no Lasso.

Conte também **quantos coeficientes cada um deixou diferentes de zero** — é a
diferença que a lista teórica previu na aritmética.

In [ ]:
for nome, modelo in [("Ridge", skl.Ridge(alpha=10.0)),                   # (a)
                     ("Lasso", skl.Lasso(alpha=1.0, max_iter=20000))]:   # (b)
    ajuste = tubo(modelo).fit(X_tr, y_tr)
    coef = ajuste.named_steps["mod"].coef_
    nao_nulos = int(np.sum(np.abs(coef) > 1e-10))                        # (c)
    print(f"{nome}  R2 treino {ajuste.score(X_tr, y_tr):.4f}   "
          f"R2 teste {ajuste.score(X_te, y_te):.4f}   "
          f"coefs != 0: {nao_nulos}/81")

Deve imprimir:

```
Ridge  R2 treino 0.7786   R2 teste 0.6363   coefs != 0: 81/81
Lasso  R2 treino 0.7080   R2 teste 0.6343   coefs != 0: 13/81
```

Três leituras:

1. **O $R^2$ de teste saltou de $-19{,}09$ para $0{,}64$.** Mesmos dados, mesmo
   número de covariáveis — só uma penalização.
2. **O $R^2$ de treino caiu** (0,9582 → 0,7786 → 0,7080). É o que se espera: a
   penalização impede o ajuste de perseguir o ruído, e o preço é errar mais no
   treino. Quem olha só o treino conclui que pioramos o modelo.
3. **O Lasso chega ao mesmo desempenho da Ridge usando 13 covariáveis em vez de
   81.** Foi a aritmética do Exercício 3 da lista teórica: a Ridge encolhe
   proporcionalmente e nunca zera, o Lasso subtrai uma constante e aniquila os
   pequenos.

> **Sua vez.** Repita o Exercício 2 e este, trocando `train_size=100` por
> `train_size=2000`. O MQO continua quebrado? E a vantagem do Lasso sobre ele,
> continua existindo?

---
## Exercício 4 — o caminho do Lasso e a escolha de $\alpha$

O $\alpha=1$ do exercício anterior foi um chute. Percorra uma grade de $\alpha$ e
guarde, para cada um, quantos coeficientes sobrevivem e qual o $R^2$ de teste.

In [ ]:
alphas = np.logspace(-2, 1.5, 15)                             # (a) de 0,01 a ~31,6
n_coefs, r2_teste = [], []

for a in alphas:
    ajuste = tubo(skl.Lasso(alpha=a, max_iter=20000)).fit(X_tr, y_tr)
    coef = ajuste.named_steps["mod"].coef_
    n_coefs.append(int(np.sum(np.abs(coef) > 1e-10)))
    r2_teste.append(ajuste.score(X_te, y_te))                 # (b)

for a, k, r2 in zip(alphas, n_coefs, r2_teste):
    print(f"alpha {a:8.4f}:  {k:2d} coefs,  R2 teste {r2:7.4f}")

Deve imprimir a grade inteira. Os pontos que importam:

| $\alpha$ | 0,0100 | 0,1000 | 0,3162 | 0,5623 | 1,0000 | 10,0000 | 31,6228 |
|---|---|---|---|---|---|---|---|
| coefs | 66 | 39 | 27 | 21 | 13 | 2 | 0 |
| $R^2$ | −1,9696 | 0,5875 | 0,6299 | **0,6512** | 0,6343 | 0,3991 | −0,0142 |

Nos dois extremos o modelo é ruim por motivos opostos: com $\alpha$ pequeno ele
volta a ser o MQO quebrado, e com $\alpha$ grande zera tudo e vira o preditor
constante ($R^2\approx 0$). O melhor da grade está em $\alpha=0{,}5623$, com 21
covariáveis.

Desenhe o caminho. Como os $\alpha$ variam em ordens de grandeza, o eixo
horizontal precisa ser logarítmico.

In [ ]:
fig, (ax1, ax2) = subplots(1, 2, figsize=(9, 3.2))

ax1.plot(alphas, n_coefs, "o-", ms=4)
ax1.set_xscale("log")                                         # (a)
ax1.set_xlabel(r"$\alpha$")
ax1.set_ylabel("coeficientes diferentes de zero")

ax2.plot(alphas, r2_teste, "o-", ms=4)
ax2.set_xscale("log")
ax2.set_ylim(-0.2, 0.75)                                      # (b) corta o -1,97
ax2.set_xlabel(r"$\alpha$")
ax2.set_ylabel("$R^2$ de teste")

fig.tight_layout()

Agora escolha o $\alpha$ **sem olhar o teste** — que é a única forma honesta.
Use validação cruzada de 5 dobras sobre o treino.

In [ ]:
busca = skm.GridSearchCV(
    tubo(skl.Lasso(max_iter=20000)),
    {"mod__alpha": alphas},                                   # (a) o nome do passo, dois underscores
    cv=skm.KFold(5, shuffle=True, random_state=0),
    scoring="neg_mean_squared_error",                         # (b)
).fit(X_tr, y_tr)

escolhido = busca.best_params_["mod__alpha"]
melhor = busca.best_estimator_
coef = melhor.named_steps["mod"].coef_

print(f"CV escolheu alpha = {escolhido:.4f}")
print(f"  coeficientes != 0: {int(np.sum(np.abs(coef) > 1e-10))}/81")
print(f"  R2 de teste:       {melhor.score(X_te, y_te):.4f}")

Deve imprimir `alpha = 0.3162`, `27/81` coeficientes e `R2 de teste 0.6299`.

A validação cruzada não acertou o melhor $\alpha$ da grade (era 0,5623, com
$R^2=0{,}6512$) — ela ficou uma casa à esquerda e perdeu 0,02 de $R^2$. Isso é o
normal, e é a coisa certa a reportar: a CV escolhe **sem ver o teste**, com 100
observações e 5 dobras, ou seja, com 80 observações por ajuste. O erro dela é
pequeno perto do que se ganhou saindo do MQO.

O que **não** se pode fazer é o que o Exercício 4 fez de propósito: olhar a
coluna de $R^2$ de teste e escolher o melhor. Aquele $0{,}6512$ não é uma
estimativa honesta de desempenho — é o máximo de 15 tentativas medidas no mesmo
conjunto.

Por fim, veja **quais** covariáveis o Lasso manteve, ordenadas por magnitude.

In [ ]:
nomes = np.array(X_todos.columns)
ordem = np.argsort(-np.abs(coef))[:5]                         # (a) as 5 de maior módulo

for j in ordem:
    print(f"{nomes[j]:42s} {coef[j]:8.3f}")

Deve imprimir:

```
wtd_std_atomic_mass                         -16.552
wtd_mean_ThermalConductivity                 13.995
range_atomic_radius                          12.020
std_atomic_mass                              10.520
range_atomic_mass                             8.135
```

Note que as covariáveis são padronizadas, então os coeficientes são comparáveis
entre si: cada um diz quantos kelvin a temperatura crítica muda quando aquela
covariável sobe um desvio-padrão, com as demais fixas.

Um alerta que vale para o curso todo: o Lasso escolheu **uma** entre várias
covariáveis correlacionadas, e essa escolha é instável — trocar a semente da
divisão troca parte da lista. Lasso é bom para **predizer com poucas variáveis**;
usá-lo como evidência de que *estas* variáveis são as importantes é uma leitura
mais forte do que o método sustenta.

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 2 | com $n=100$ e $d=81$, o MQO dá $R^2$ de treino 0,9582 e de teste **−19,09** |
| 3 | Ridge e Lasso levam o mesmo treino a $R^2\approx 0{,}64$ — e o Lasso usa 13 covariáveis |
| 4 | o caminho vai de 66 coeficientes ($\alpha=0{,}01$) a nenhum ($\alpha=31{,}6$) |
| 4 | a CV escolhe $\alpha=0{,}3162$ e perde 0,02 de $R^2$ para o melhor da grade — sem olhar o teste |

**A seguir.** A Aula 03 formaliza o que o último exercício fez à mão: como estimar
risco e escolher hiperparâmetro sem gastar um conjunto de teste, e como reportar
desempenho depois de ter escolhido.